In [ ]:
from pathlib import Path
import re
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib import ticker
from netCDF4 import Dataset
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import wasserstein_distance
from scipy.integrate import solve_ivp

In [ ]:
target_dir = Path.cwd().parent / "model_training"

# Add the absolute path to sys.path
sys.path.append(str(target_dir.resolve()))

# Add the parent directory to sys.path
sys.path.append(str(Path.cwd().parent))
from models import load_model_checkpoint, do_inference
from physics import set_parameter_ranges, dydt, scale_droplet_parameters

In [ ]:
# OPTIONS
history_file_count           = 10
history_record_interval      = 10       # Records how many seconds between history file writes
steady_state_cutoff          = 8000.0
steady_state_cutoff_index    = int( steady_state_cutoff // history_record_interval + 1 )
pre_equilibrium_cutoff       = 800.0
pre_equilibrium_cutoff_index = int( pre_equilibrium_cutoff // history_record_interval + 1 )

#per_layer_margins = {
#    "RHxym"  : ("absolute", 0.005), 
#    "qstarm" : ("relative", 0.001),
#    "Tpmean" : ("absolute", 0.01),
#    "Hpsrc"  : ("absolute", 1.00e-7), 
#    "Tpsrc"  : ("absolute", 2.5e-4),
#    "ql"     : ("absolute", 2.5e-6)
#}

per_layer_dynamic_range_tost_tolerance = 0.0025 # Percent of the range of a variable over time/layer/ensembles
per_layer_dynamic_range_tost_window    = [0, 127]
scalar_dynamic_range_tost_tolerance    = 0.015 # Percent of the range of a variable over time/layer/ensembles


# 1. Systematically set large, consistent font sizes
LARGE_FONTS = {
    'font.size': 14,                     # Global default fallback
    'axes.labelsize': 18,                # X and Y axis labels
    'axes.titlesize': 20,                # Subplot titles
    'xtick.labelsize': 15,               # X-tick labels
    'ytick.labelsize': 15,               # Y-tick labels
    'legend.fontsize': 14,               # Legend text
    'figure.titlesize': 22,              # Global figure suptitle
    'axes.formatter.useoffset': False    # Turn off y-axis offset
}
plt.rcParams.update(LARGE_FONTS)

# Define base dimensions for a SINGLE plot panel
base_width  = 6
base_height = 4.5
base_size   = (base_width, base_height)

In [ ]:
device     = "cpu"
model_path = "../model_training/models/flippant-gusto.pt"

# ---------------------------------

mlp_history_folder     = "paper_data/history/mlp/"
be_history_folder      = "paper_data/history/be/"

# Folder structure for multiple history file analysis:
#   be/
#     1/
#       history.nc
#       histograms.nc
#     ...
#     10/
#       history.nc
#       histograms.nc
#   mlp/
#     1/
#       history.nc
#       histograms.nc
#     ...
#     10/
#       history.nc
#       histograms.nc
# The following generates the corresponding file names:

history_filenames = {
    "be":  [f"{be_history_folder:s}/{file_index:d}/history.nc" for file_index in range(1,history_file_count + 1)],
    "mlp": [f"{mlp_history_folder:s}/{file_index:d}/history.nc" for file_index in range(1,history_file_count + 1)]
}
histogram_filenames = {
    "be":  [f"{be_history_folder:s}/{file_index:d}/histograms.nc" for file_index in range(1,history_file_count + 1)],
    "mlp": [f"{mlp_history_folder:s}/{file_index:d}/histograms.nc" for file_index in range(1,history_file_count + 1)]
}

history_variable_names = {
    "auxiliary" : ["time", "zconc", "tnumpart", "dt", "zu"],
    "averaged"  : ["Tpmean", "Tfmean", "Tpmsqr"],
    "scalar"    : ["meanRH", "radavg", "tnumdrop", "radmsqr", "varRH"],
    "per_layer" : ["radmean", "Tpmean", "RHxym", "Tpsrc", "Hpsrc", "qstarm", "ql", "txym"]
}

history_timeseries_variable_types = ["averaged", "scalar", "per_layer"]  # Which variables to run time series analysis on

history_variable_labels = {
    "time"     : "Time",
    "zconc"    : "Droplet Z Concentration",
    "tnumpart" : "\\text{Total Particle Count}",
    "dt"       : "Time Step",
    "zu"       : "Height",
    "Tpmean"   : "\\langle T_p\\rangle_{xyz}",
    "Tfmean"   : "\\langle T_f\\rangle_{xyz}",
    "meanRH"   : "\\langle RH\\rangle_{xyz}",
    "radmean"  : "\\langle r_p \\rangle_{xyt}",
    "txym"     : "\\langle q_v \\rangle_{xyt}",
    "radavg"   : "\\langle r_p\\rangle_{xyz}",
    "radmsqr"  : "\\langle r_p^2\\rangle_{xyz}",
    "varRH"    : "Var(RH)_{xyz}",
    "varrad"   : "Var(r_p)_{xyz}",
    "Tpmsqr"   : "Var(T_p)_{xyz}",
    "tnumdrop" : "\\langle N_p \\rangle_{}",
    "RHxym"    : "\\langle RH\\rangle_{xyt}",
    "qstarm"   : "\\langle q_* \\rangle_{xyt}",  # "Average Vapor Pressure at Droplet Surface",
    "Hpsrc"    : "\\langle Hp_{src} \\rangle_{xyt}",
    "Tpsrc"    : "\\langle Tp_{src} \\rangle_{xyt}",
    "ql"       : "\\langle q_l \\rangle_{xyt}"
}

history_variable_units = {
    "time"     : "s",
    "zconc"    : "$\\text{droplets}~m^{-3}$",
    "tnumpart" : "#",
    "dt"       : "s",
    "zu"       : "m",
    "Tpmean"   : "K",
    "Tfmean"   : "K",
    "varRH"    : "\\%^2",
    "varTp"    : "K^2",
    "varrad"   : "(\\mu m)^2",
    "meanRH"   : "\\%",
    "radmean"  : "\\mu m",
    "txym"     : "g~kg^{-1}",
    "radavg"   : "\\mu m",
    "radmsqr"  : "(\\mu m)^2",
    "RHmsqr"   : "\\%",
    "tnumdrop" : "",
    "RHxym"    : "\\%",
    "Tpmsqr"   : "K^2",
    "qstarm"   : "g~kg^{-1}",
    "Hpsrc"    : "kg~kg^{-1}~s^{-1}",
    "Tpsrc"    : "K~s^{-1}",
    "ql"       : "g~kg^{-1}"
}

In [ ]:
history_files = {
    "be"  : [Dataset(filename, mode='a') for filename in history_filenames["be"]],
    "mlp" : [Dataset(filename, mode='a') for filename in history_filenames["mlp"]] 
}
histogram_files = {
    "be"  : [Dataset(filename, mode='a') for filename in histogram_filenames["be"]],
    "mlp" : [Dataset(filename, mode='a') for filename in histogram_filenames["mlp"]] 
}

In [ ]:
def _plot_defaults( fig=None, ax=None ):
    if ax is None:
        fig = plt.figure()
        ax  = fig.gca()
    ax.minorticks_on()
    ax.grid( color="k", alpha=0.1 )

    return fig, ax

def _multiplot_defaults( shape ):
    fig, ax_h = plt.subplots( *shape, constrained_layout=True )
    for ax in ax_h.flat:
        _plot_defaults( ax=ax )

    return fig, ax_h

def history_per_layer_ribbon_plot( history_data, variable_name, layer_window=[0,128], ax=None, **kwargs ):
    be_data    = history_data["be"]["per_layer"][variable_name][..., layer_window[0]:layer_window[1]]
    mlp_data   = history_data["mlp"]["per_layer"][variable_name][..., layer_window[0]:layer_window[1]]
    data_label = "$" + history_variable_labels[variable_name] + "$"
    data_unit  = "$" + history_variable_units[variable_name] + "$"

    indices = history_data["be"]["auxiliary"]["zu"][0][layer_window[0]:layer_window[1]]

    ax = ribbon_plot( be_data, mlp_data, data_label, data_unit, indices=indices, vertical=True, ax=ax, **kwargs )
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=7))

    return ax

def ribbon_plot( be_data, mlp_data, data_label, data_unit, ax=None, indices=None, vertical=False, reduce_dimensions=(0, 1), std_a=None, std_b=None, rmse=False, rmse_relocate=False ):
    # We average over runs (dim 0) and time (dim 1) to get a mean per vertical index
    mean_a = np.mean(be_data, axis=reduce_dimensions)
    
    mean_b = np.mean(mlp_data, axis=reduce_dimensions)

    if std_a is None:
        std_a  = np.std(be_data, axis=reduce_dimensions)
    if std_b is None:
        std_b  = np.std(mlp_data, axis=reduce_dimensions)

    if rmse:
        rmse_data = np.sqrt( np.sum( ( mean_a - mean_b ) ** 2 )/mean_a.shape[0] )
    
    if indices is None:
        indices = np.arange(be_data.shape[-1])  # 0 to 127

    # 2. Setup Plot Aesthetics
    plt.style.use('seaborn-v0_8-muted') # Uses a clean, modern aesthetic
    if ax is None:
        fig, ax = plt.subplots(figsize=base_size)
    
    # Define Colors
    color_a = '#3498db'
    color_b = '#e67e22'

    if vertical:
        # 3. Plotting Group A
        ax.plot(mean_a, indices, color=color_a, lw=2, label='BE Mean', zorder=3)
        ax.fill_betweenx(indices, mean_a - std_a, mean_a + std_a, 
                        color=color_a, alpha=0.15, label=f'BE Mean ±1σ', zorder=2)
    
        # 4. Plotting Group B
        ax.plot(mean_b, indices, color=color_b, lw=2, label=f'MLP Mean', zorder=3)
        ax.fill_betweenx(indices, mean_b - std_b, mean_b + std_b, 
                        color=color_b, alpha=0.15, label=f'MLP Mean ±1σ', zorder=2)

        ax.set_ylabel('Height [m]' )
        ax.set_xlabel(f'{data_label} [{data_unit}]' )
    else:
        # 3. Plotting Group A
        ax.plot(indices, mean_a, color=color_a, lw=2, label='BE Mean', zorder=3)
        ax.fill_between(indices, mean_a - std_a, mean_a + std_a, 
                        color=color_a, alpha=0.15, label=f'BE Mean ±1σ', zorder=2)
    
        # 4. Plotting Group B
        ax.plot(indices, mean_b, color=color_b, lw=2, label=f'MLP Mean', zorder=3)
        ax.fill_between(indices, mean_b - std_b, mean_b + std_b, 
                        color=color_b, alpha=0.15, label=f'MLP Mean ±1σ', zorder=2)

        ax.set_xlabel('Height [m]' )
        ax.set_ylabel(f'{data_label} [{data_unit}]' )
    
    if rmse:
        if rmse_relocate:
            ax.text(0.95, 0.165, f'RMSE: {rmse_data:.3e} [{data_unit}]', transform=ax.transAxes, ha='right', va='center', weight='bold', fontsize=14)
        else:
            ax.text(0.05, 0.10, f'RMSE: {rmse_data:.3e} [{data_unit}]', transform=ax.transAxes, ha='left', va='center', weight='bold', fontsize=14)
        

    # 5. Professional Formatting
    #ax.set_title(f"Vertical Profile Comparison for {data_label} in BE vs. MLP", fontsize=16, fontweight='bold', pad=20, family='sans-serif')
    
    # Grid and Spines
    ax.minorticks_on()
    ax.grid(True, linestyle='--', alpha=0.4, zorder=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout(pad=0.25)
    # Legend
    ax.legend(loc='best', frameon=True, facecolor='white', framealpha=0.9)

    return ax

def ribbon_plot_error( be_data, mlp_data, data_label, data_unit, ax=None, indices=None, vertical=False, reduce_dimensions=(0, 1), ci=None ):
    # We average over runs (dim 0) and time (dim 1) to get a mean per vertical index
    mean_a = np.mean(be_data, axis=reduce_dimensions)
    mean_b = np.mean(mlp_data, axis=reduce_dimensions)

    error_in_mean = mean_b - mean_a

    if ci is None:
        # Perform Welch's independent t-test along the runs axis (axis=0)
        ci = stats.ttest_ind(mlp_data, be_data, axis=reduce_dimensions, equal_var=False).confidence_interval( confidence_level=0.95 )
        ci = np.stack( [
            ci.low, ci.high,
        ], axis=-1 )

    #std           = np.std( error_in_mean, axis=reduce_dimensions )
    
    if indices is None:
        indices = np.arange(be_data.shape[-1])  # 0 to 127

    # 2. Setup Plot Aesthetics
    plt.style.use('seaborn-v0_8-muted') # Uses a clean, modern aesthetic
    if ax is None:
        fig, ax = plt.subplots(figsize=base_size)
    
    # Define Colors
    color_a = '#3498db'
    color_b = '#e67e22'

    if vertical:
        ax.plot(error_in_mean, indices, color=color_b, lw=2, label='(MLP - BE) Mean', zorder=3)
        
        ax.axvline( x=0.0, color='k', lw=1.5, linestyle="--", alpha=0.6 )
        #if std:
        #    ax.fill_betweenx(indices, -std_a, std_a, 
        #                    color=color_a, alpha=0.15, label=f'±1σ BE Spread', zorder=2)
        #    ax.fill_betweenx(indices, -std_b, std_b, 
        #                    color=color_b, alpha=0.15, label=f'±1σ MLP Spread', zorder=2)
        #else:
        ax.fill_betweenx(indices, ci[:, 0], ci[:, 1],
                        color=color_b, alpha=0.15, label=f'±95% CI', zorder=2)
    
        ax.set_ylabel('Z Position [m]'  )
        ax.set_xlabel(f'{data_label} [{data_unit}]' )
    else:
        ax.plot( indices, error_in_mean, color=color_b, lw=2, label='(MLP - BE) Mean', zorder=3)

        ax.axhline( y=0.0, color='k', lw=1.5, linestyle="--", alpha=0.6, xmax=indices[-1] )
        #if std:
        #    ax.fill_between(indices, -std_a, std_a, 
        #                    color=color_a, alpha=0.15, label=f'±1σ BE Spread', zorder=2)
        #    ax.fill_between(indices, -std_b, std_b, 
        #                    color=color_b, alpha=0.15, label=f'±1σ MLP Spread', zorder=2)

        ax.fill_between(indices, ci[:, 0], ci[:, 1], 
                        color=color_b, alpha=0.15, label=f'±95% CI', zorder=2)
    
        ax.set_xlabel('Z Position [m]' )
        ax.set_ylabel(f'{data_label} [{data_unit}]' )

    # 5. Professional Formatting
    #ax.set_title(f"Vertical Profile Comparison for {data_label} in BE vs. MLP", fontsize=16, fontweight='bold', pad=20, family='sans-serif')
    
    # Grid and Spines
    ax.minorticks_on()
    ax.grid(True, linestyle='--', alpha=0.4, zorder=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout(pad=0.25)
    # Legend
    ax.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9)

    return ax

In [ ]:
# Create folders
import os

os.makedirs("../paper_figures", exist_ok=True)
os.makedirs("../paper_figures/loss", exist_ok=True)
os.makedirs("../paper_figures/histograms", exist_ok=True)
os.makedirs("../paper_figures/per_layer", exist_ok=True)
os.makedirs("../paper_figures/time_series", exist_ok=True)
os.makedirs("../paper_figures/performance", exist_ok=True)
os.makedirs("../paper_figures/error_analysis_figures", exist_ok=True)

In [ ]:
paper_training_data      = pd.read_csv( "paper_data/training/training_loss.csv" )
paper_validation_data    = pd.read_csv( "paper_data/training/validation_loss.csv" )
paper_learning_rate_data = pd.read_csv( "paper_data/training/learning_rate.csv" )

paper_training_data

In [ ]:

paper_model_names = [
    "obtainable-policy",
    "unique-reaction",
    "yearly-intermission",
    "flippant-gusto",
    "marked-joseph",
]

# Slot order maximizes the minimum adjacent colorblind separation (worst adjacent
# tritan dE 32.9); each model also carries a dash pattern so identity survives
# grayscale printing.
paper_model_colors = {
    "obtainable-policy": "#eb6834",
    "unique-reaction": "#2a78d6",
    "yearly-intermission": "#e34948",
    "flippant-gusto": "#4a3aa7",
    "marked-joseph": "#008300",
}
paper_model_dashes = {
    "obtainable-policy": (None, None),
    "unique-reaction": (6, 1.5),
    "yearly-intermission": (1.5, 1.5),
    "flippant-gusto": (6, 1.5, 1.5, 1.5),
    "marked-joseph": (3, 1.5),
}

# Peak learning rate of each schedule, as unicode superscripts rather than mathtext
# so the labels display cleanly in the legend and in plain text. U+00B7 stands in
# as the superscript decimal point -- Unicode has no superscript full stop.
paper_model_learning_rates = {
    "obtainable-policy": "10⁻¹·⁵",
    "unique-reaction": "10⁻²",
    "yearly-intermission": "10⁻²·⁵",
    "flippant-gusto": "10⁻³",
    "marked-joseph": "10⁻³·⁵"
}

# Number of logged points to smooth the raw training loss over. The raw trace is
# kept underneath at low alpha so the noise floor stays visible.
training_smoothing_window = 51


def _paper_loss_defaults( ax ):
    ax.set_yscale( "log" )
    ax.set_xlabel( "Batch Index" )
    ax.minorticks_on()
    ax.grid( True, which="major", linestyle="--", alpha=0.4, zorder=1 )
    ax.grid( True, which="minor", axis="y", linestyle=":", alpha=0.2, zorder=1 )
    ax.spines["top"].set_visible( False )
    ax.spines["right"].set_visible( False )
    ax.ticklabel_format( axis="x", style="sci", scilimits=(0, 0), useMathText=True )


plt.style.use( "seaborn-v0_8-muted" )
plt.rcParams.update( LARGE_FONTS )  # style.use() resets the font sizes set in the options cell

fig, ax_h = plt.subplots( nrows=1, ncols=3, figsize=(base_width*3, base_height) )

training_batches = paper_training_data["batch"].to_numpy()
training_steps   = paper_training_data["flippant-gusto - _step"].to_numpy()

validation_epochs    = paper_validation_data["epoch"].to_numpy()
learning_rate_epochs = paper_learning_rate_data["epoch"].to_numpy()

for model_name in paper_model_names:
    color = paper_model_colors[model_name]
    dash  = paper_model_dashes[model_name]
    label = f"η = {paper_model_learning_rates[model_name]}"

    training_loss = paper_training_data[f"{model_name} - training_loss"]
    smoothed_loss = training_loss.rolling(
        window=training_smoothing_window, center=True, min_periods=1
    ).median()

    ax_h[0].plot( training_batches, training_loss,
                  color=color, lw=0.7, alpha=0.15, zorder=2 )
    ax_h[0].plot( training_batches, smoothed_loss,
                  color=color, lw=2, dashes=dash, label=label, zorder=3 )

    ax_h[1].plot( validation_epochs, paper_validation_data[f"{model_name} - validation_loss"],
                  color=color, lw=2, dashes=dash, marker="o", markersize=5,
                  markeredgecolor="white", markeredgewidth=0.6,
                  label=label, zorder=3 )

    # The __MIN/__MAX columns here are wandb's within-bucket range from downsampling,
    # not a run-to-run spread, so only the mean schedule is drawn.
    ax_h[2].plot( learning_rate_epochs, paper_learning_rate_data[f"{model_name} - learning_rate"],
                  color=color, lw=2, dashes=dash, label=label, zorder=3 )


for axis_index, ax in enumerate( ax_h.flat ):
    ax.text(0.12, 0.05, f'({chr( ord( 'A' ) + axis_index)})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)

_paper_loss_defaults( ax_h[0] )
ax_h[0].set_ylabel( "Training Loss" )
ax_h[0].legend( loc="upper right", frameon=True, facecolor="white", framealpha=0.9 )

_paper_loss_defaults( ax_h[1] )
ax_h[1].set_ylabel( "Validation Loss" )
ax_h[1].set_xlabel( "Epoch" )

_paper_loss_defaults( ax_h[2] )
ax_h[2].set_ylabel( "Learning Rate" )
ax_h[2].set_xlabel( "Epoch" )

#ax_h[2].set_yscale( "linear" )

fig.align_ylabels()
plt.tight_layout()

os.makedirs( "../paper_figures/loss", exist_ok=True )
plt.savefig( "../paper_figures/loss/training_validation_loss.pdf",
             bbox_inches="tight",
             dpi=300 )

In [ ]:
def compute_integral_time_scale(x):
    """Computes the integral time scale (in sample units) using the first zero-crossing."""
    # 1. Standardize and compute autocorrelation function (ACF)
    x_centered = x - np.mean(x)
    autocorr = np.correlate(x_centered, x_centered, mode='full')
    autocorr = autocorr[len(x)-1:] / autocorr[len(x)-1]
    
    # 2. Find the first zero-crossing
    zero_crossings = np.where(autocorr < 0)[0]
    k_0 = zero_crossings[0] if len(zero_crossings) > 0 else len(autocorr)
    
    # 3. Integrate (sum) up to the zero-crossing
    tau_int = 0.5 + np.sum(autocorr[1:k_0])
    return tau_int

In [ ]:
def compute_ensemble_integral_time_scale(data, convention='statistics'):
    """
    Computes the integrated autocorrelation time for multiple parallel time series.
    
    Parameters:
    -----------
    data : 2D numpy array of shape (M, N)
        M = number of parallel realizations/trajectories
        N = length of each time series
    convention : 'statistics' or 'fluid'
        'statistics' returns tau matching emcee convention (1 + 2 * sum)
        'fluid' returns T matching fluid dynamics convention (0.5 + sum)
    """
    M, N = data.shape
    
    # 1. Center each time series individually (subtract row means)
    data_centered = data - np.mean(data, axis=1, keepdims=True)
    
    # 2. Compute normalized ACF for each series
    acfs = np.zeros((M, N))
    for i in range(M):
        x = data_centered[i]
        corr = np.correlate(x, x, mode='full')
        corr = corr[N-1:]  # Keep non-negative lags
        acfs[i] = corr / corr[0]  # Normalize so lag 0 equals 1.0
        
    # 3. Average the ACFs across all realizations
    mean_acf = np.mean(acfs, axis=0)
    
    # 4. Find the first zero-crossing of the ENSEMBLE ACF
    zero_crossings = np.where(mean_acf < 0)[0]
    k_0 = zero_crossings[0] if len(zero_crossings) > 0 else len(mean_acf)
    
    # 5. Integrate up to the first zero-crossing
    if convention == 'statistics':
        tau = 1.0 + 2.0 * np.sum(mean_acf[1:k_0])
    else:  # 'fluid'
        tau = 0.5 + np.sum(mean_acf[1:k_0])
        
    return tau

def compute_ensemble_integral_time_scale_fast(data, convention='statistics'):
    """
    Computes the integrated autocorrelation time for multiple parallel time series.
    Vectorized over all realizations using FFT for O(N log N) performance.
    
    Parameters:
    -----------
    data : 2D numpy array of shape (M, N)
        M = number of parallel realizations/trajectories
        N = length of each time series
    convention : 'statistics' or 'fluid'
        'statistics' returns tau matching emcee convention (1 + 2 * sum)
        'fluid' returns T matching fluid dynamics convention (0.5 + sum)
    """
    M, N = data.shape
    
    # 1. Center the data across the time axis (subtract the mean of each row)
    data_centered = data - np.mean(data, axis=1, keepdims=True)
    
    # 2. Vectorized Autocorrelation using FFT
    # We pad the time series to 2*N to avoid circular correlation wrapping
    F = np.fft.rfft(data_centered, n=2*N, axis=1)
    
    # Power Spectral Density (PSD) is the square of the magnitude
    PSD = np.abs(F)**2
    
    # Inverse FFT to return to the time domain, keeping only the first N positive lags
    acfs = np.fft.irfft(PSD, n=2*N, axis=1)[:, :N]
    
    # 3. Normalize each ACF by its variance (the lag-0 value)
    # Using slicing [:, 0:1] keeps the 2D shape for proper broadcasting
    acfs_normalized = acfs / acfs[:, 0:1]
    
    # 4. Ensemble average the ACFs across all realizations
    mean_acf = np.mean(acfs_normalized, axis=0)
    
    # 5. Find the first zero-crossing of the ensemble ACF
    zero_crossings = np.where(mean_acf < 0)[0]
    k_0 = zero_crossings[0] if len(zero_crossings) > 0 else N
    
    # 6. Integrate up to the first zero-crossing
    if convention == 'statistics':
        tau = 1.0 + 2.0 * np.sum(mean_acf[1:k_0])
    else:  # 'fluid'
        tau = 0.5 + np.sum(mean_acf[1:k_0])
        
    return tau

## History Analysis

In [ ]:
history_variables                = {}

# pull all history variables
for backend in ["be", "mlp"]:
    history_variables[backend] = {}
    for variable_type, variable_names in history_variable_names.items():
        history_variables[backend][variable_type] = {
            variable_name: [file.variables[variable_name][:] for file in history_files[backend]]
                for variable_name in variable_names
        }

In [ ]:
max_time = min( np.max( [timeline[-1] for timeline in history_variables["be"]["auxiliary"]["time"]] ),
                np.max( [timeline[-1] for timeline in history_variables["be"]["auxiliary"]["time"]] ) )

max_iteration = np.min( [np.searchsorted( timeline, max_time )
                        for timeline in [*history_variables["be"]["auxiliary"]["time"],
                                         *history_variables["mlp"]["auxiliary"]["time"]]] )

min_iteration = np.max( [np.searchsorted( timeline, steady_state_cutoff )
                         for timeline in [*history_variables["be"]["auxiliary"]["time"],
                                          *history_variables["mlp"]["auxiliary"]["time"]]] )

# Refactor Variables so that they're homogenous numpy arrays over each file
#     and all taken in steady state (i.e. past 8000s) -- steady state cuttoff
#     is applied after plotting of initial state
# and average "averaged" variables by layer
for backend in ["be", "mlp"]:
    for variable_type, variable_names in history_variable_names.items():
        for variable_name in variable_names:
            # Keep "zu" unaffected...
            if variable_name == "zu":
                continue
            history_variables[backend][variable_type][variable_name] = np.array(
                [per_simulation_variable[0:max_iteration]
                 for per_simulation_variable in history_variables[backend][variable_type][variable_name]]
            )
            if variable_type == "averaged":
                layer_volumes       = 4*(np.diff(history_files["be"][0].variables["zw"][:])) # NOTE: Assumes all runs have the SAME zw
                layer_concentration = history_variables[backend]["auxiliary"]["zconc"]
                overall_count       = history_variables[backend]["auxiliary"]["tnumpart"]

                history_variables[backend][variable_type][variable_name] = (
                    history_variables[backend][variable_type][variable_name] * layer_volumes * layer_concentration).sum( axis=2 ) / overall_count

In [ ]:
# Scale all radius data and calculate variance
if history_variables["be"]["scalar"]["radavg"][0][0] < 1.0e-4:
    history_variables["be"]["scalar"]["radavg"]  *= 1.0e6
    history_variables["mlp"]["scalar"]["radavg"]  *= 1.0e6

    history_variables["be"]["scalar"]["radmsqr"] *= 1.0e12
    history_variables["mlp"]["scalar"]["radmsqr"] *= 1.0e12

    history_variables["be"]["per_layer"]["radmean"] *= 1.0e6
    history_variables["mlp"]["per_layer"]["radmean"] *= 1.0e6

    history_variables["be"]["scalar"]["varrad"]  = history_variables["be"]["scalar"]["radmsqr"] - history_variables["be"]["scalar"]["radavg"] ** 2 
    history_variables["mlp"]["scalar"]["varrad"] = history_variables["mlp"]["scalar"]["radmsqr"] - history_variables["mlp"]["scalar"]["radavg"] ** 2


# scale txym, ql, qstarm
if history_variables["be"]["per_layer"]["ql"][0, -1, 0] < 1.0e-2:
    history_variables["be"]["per_layer"]["ql"]  *= 1.0e2
    history_variables["be"]["per_layer"]["txym"]  *= 1.0e2
    history_variables["be"]["per_layer"]["qstarm"]  *= 1.0e2

    history_variables["mlp"]["per_layer"]["ql"]  *= 1.0e2
    history_variables["mlp"]["per_layer"]["txym"]  *= 1.0e2
    history_variables["mlp"]["per_layer"]["qstarm"]  *= 1.0e2



In [ ]:
if len( history_variables["be"]["per_layer"]["txym"].shape ) == 4:
    history_variables["be"]["per_layer"]["txym"] = history_variables["be"]["per_layer"]["txym"][:, :, 1, :]

if len( history_variables["mlp"]["per_layer"]["txym"].shape ) == 4:
    history_variables["mlp"]["per_layer"]["txym"] = history_variables["mlp"]["per_layer"]["txym"][:, :, 1, :]

In [ ]:
initial_plots = [
    ("scalar", "radavg"),
    ("averaged", "Tpmean"),
    ("scalar", "meanRH"),
]

fig, ax_h = plt.subplots( nrows=3, ncols=2, figsize=(base_width*2,base_height*3), sharex=True )

r = 0
for variable_type, variable_name in initial_plots:
        ax = ribbon_plot( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index] ,
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                "$" + history_variable_labels[variable_name] + "$",
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r][0],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0),
                rmse=True,
                rmse_relocate=True
        )

        if r > 0:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r == 2:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )

        ax.text(0.95, 0.10, f'({chr( ord( 'A' ) + 2*r )})', transform=ax.transAxes, ha='right', va='center', weight='bold', fontsize=14)

        ax = ribbon_plot_error( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                "$" + history_variable_labels[variable_name] + "$",
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r][1],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0)
        )
        ax.set_xlabel( "Time [s]" )
        ax.set_ylabel( f"${history_variable_labels[variable_name]}$ Error $[{history_variable_units[variable_name]}]$" )

        if r > 0:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r == 2:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )

        ax.text(0.95, 0.10, f'({chr( ord( 'A' ) + 2*r + 1 )})', transform=ax.transAxes, ha='right', va='center', weight='bold', fontsize=14)
 
        r += 1

fig.align_ylabels()
plt.tight_layout()
plt.savefig( f"../paper_figures/time_series/be_mlp_time_series_initial.pdf",
             bbox_inches="tight",
             dpi=300 )

In [ ]:
initial_plots = [
    ("scalar", "varrad"),
    ("averaged", "Tpmsqr"),
    ("scalar", "varRH"),
]

fig, ax_h = plt.subplots( nrows=3, ncols=2, figsize=(base_width*2,base_height*3), sharex=True )

r = 0
for variable_type, variable_name in initial_plots:
        ax = ribbon_plot( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index] ,
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                "$" + history_variable_labels[variable_name] + "$",
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r][0],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0),
                rmse=True,
                rmse_relocate=True
        )

        if r > 0:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r == 2:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )

        ax.text(0.95, 0.10, f'({chr( ord( 'A' ) + 2*r )})', transform=ax.transAxes, ha='right', va='center', weight='bold', fontsize=14)
         
        ax = ribbon_plot_error( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                history_variable_labels[variable_name],
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r][1],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0)
        )
        ax.set_xlabel( "Time [s]" )
        ax.set_ylabel( f"${history_variable_labels[variable_name]}$ Error [${history_variable_units[variable_name]}$]" )

        if r > 0:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r == 2:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )

        ax.text(0.95, 0.10, f'({chr( ord( 'A' ) + 2*r + 1 )})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)
        
        r += 1

fig.align_ylabels()
plt.tight_layout()
plt.savefig( f"../paper_figures/time_series/be_mlp_time_series_initial_variance.pdf",
             bbox_inches="tight",
             dpi=300 )

In [ ]:
initial_plots = [
    ("scalar", "radavg"),
    ("scalar", "varrad"),
    ("averaged", "Tpmean"),
    ("averaged", "Tpmsqr"),
    ("scalar", "meanRH"),
    ("scalar", "varRH"),
]

fig, ax_h = plt.subplots( nrows=3, ncols=4, figsize=((base_width-2.2)*4,base_height*3), sharex=True )

r = 0
for variable_type, variable_name in initial_plots:
        ax = ribbon_plot( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index] ,
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                "$" + history_variable_labels[variable_name] + "$",
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r//2][2*(r%2)],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0)
        )

        if r != 1:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r >= 4:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )

        ax.text(0.95, 0.05, f'({chr( ord( 'A' ) + 2*r )})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)

        ax = ribbon_plot_error( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                "$" + history_variable_labels[variable_name] + "$",
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r//2][2*(r%2) + 1],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0)
        )
        ax.set_xlabel( "Time [s]" )
        ax.set_ylabel( f"${history_variable_labels[variable_name]}$ Error $[{history_variable_units[variable_name]}]$" )

        if r != 1:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r >= 4:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )


        ax.text(0.95, 0.05, f'({chr( ord( 'A' ) + 2*r + 1 )})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)
 
        r += 1

fig.align_ylabels()
plt.tight_layout()
plt.savefig( f"../paper_figures/time_series/be_mlp_time_series_initial_mean_and_variance.pdf",
             bbox_inches="tight",
             dpi=300 )

In [ ]:

initial_plots = [
    ("scalar", "radavg"),
    ("scalar", "varrad"),
    ("averaged", "Tpmean"),
    ("averaged", "Tpmsqr"),
    ("scalar", "meanRH"),
    ("scalar", "varRH"),
]

fig, ax_h = plt.subplots( nrows=3, ncols=4, figsize=((base_width-2.2)*4,base_height*3), sharex=True )

r = 0
for variable_type, variable_name in initial_plots:
        ax = ribbon_plot( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index] ,
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                "$" + history_variable_labels[variable_name] + "$",
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r//2][2*(r%2)],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0)
        )

        if r != 1:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r >= 4:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )
        
        if r % 2 > 0:
                ax.set_ylabel( "" )

        ax.text(0.95, 0.05, f'({chr( ord( 'A' ) + 2*r )})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)

        ax = ribbon_plot_error( 
                history_variables["be"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                history_variables["mlp"][variable_type][variable_name][:, :pre_equilibrium_cutoff_index],
                "$" + history_variable_labels[variable_name] + "$",
                "$" + history_variable_units[variable_name] + "$",
                ax=ax_h[r//2][2*(r%2) + 1],
                indices=history_variables["be"]["auxiliary"]["time"][0, :pre_equilibrium_cutoff_index],
                vertical=False,
                reduce_dimensions=(0)
        )
        ax.set_xlabel( "Time [s]" )
        ax.set_ylabel( f"${history_variable_labels[variable_name]}$ Error $[{history_variable_units[variable_name]}]$" )

        if r != 1:
                ax.get_legend().remove()
        else:
                ax.get_legend().set_loc( "upper right" )

        if r >= 4:
                ax.set_xlabel( "Time [s]" )
        else:
                ax.set_xlabel( "" )

        ax.set_ylabel( "" )


        ax.text(0.95, 0.05, f'({chr( ord( 'A' ) + 2*r + 1 )})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)
 
        r += 1

ax_h[0][0].set_title( "Ensemble-Mean" )
ax_h[0][1].set_title( "$\\Delta$ Ensemble-Mean" )

ax_h[0][2].set_title( "Ensemble-Variance" )
ax_h[0][3].set_title( "$\\Delta$ Ensemble-Variance" )

fig.align_ylabels()
plt.tight_layout()
plt.savefig( f"../paper_figures/time_series/be_mlp_time_series_initial_mean_and_variance_relabeled.pdf",
             bbox_inches="tight",
             dpi=300 )

In [ ]:

# Apply steady state cutoff
for backend in ["be", "mlp"]:
    for variable_type, variable_names in history_variable_names.items():
        for variable_name in variable_names:
            if variable_name == "zu":
                continue
            history_variables[backend][variable_type][variable_name] = history_variables[backend][variable_type][variable_name][:, min_iteration:]

In [ ]:

history_variable_taus = {}
for variable_type in history_timeseries_variable_types:
    print("be", variable_type)
    if variable_type == "per_layer":
        history_variable_taus[variable_type] = {
            variable_name: [int( np.ceil(  compute_ensemble_integral_time_scale_fast(
                variable_data[..., i]
            ) ) )
            for i in range( variable_data.shape[2] )]
            for variable_name, variable_data in history_variables["be"][variable_type].items()
        }
        continue
    history_variable_taus[variable_type] = {
        variable_name: int( np.ceil( compute_ensemble_integral_time_scale_fast(
            variable_data
        ) ) )
        for variable_name, variable_data in history_variables["be"][variable_type].items()
    }

In [ ]:
mlp_history_variable_taus = {}
for variable_type in history_timeseries_variable_types:
    print("mlp", variable_type)
    if variable_type == "per_layer":
        mlp_history_variable_taus[variable_type] = {
            variable_name: [int( np.ceil( compute_ensemble_integral_time_scale_fast(
                variable_data[..., i]
            ) ) )
            for i in range( variable_data.shape[2] )]
            for variable_name, variable_data in history_variables["mlp"][variable_type].items()
        }
        continue
    mlp_history_variable_taus[variable_type] = {
        variable_name: int( np.ceil( compute_ensemble_integral_time_scale_fast(
            variable_data
        ) ) )
        for variable_name, variable_data in history_variables["mlp"][variable_type].items()
    }

In [ ]:
history_variable_ranges = {}

for variable_type, variable_names in history_variable_names.items():
    history_variable_ranges[variable_type] = {}
    for variable_name in variable_names:
        if variable_type == "per_layer":
            history_variable_ranges[variable_type][variable_name] = np.ptp(
                history_variables["be"][variable_type][variable_name][..., per_layer_dynamic_range_tost_window[0]:per_layer_dynamic_range_tost_window[1]] 
            )
            continue
        history_variable_ranges[variable_type][variable_name] = np.ptp(
            history_variables["be"][variable_type][variable_name]
        )


In [ ]:
averaged_variable_types = [
    "averaged",
    "scalar",
    "per_layer"
]

# Average relavent variables on time
history_variable_time_averages = {
    backend: {
        variable_type: {
            variable_name: np.mean( history_variables[backend][variable_type][variable_name], axis=1 )
                for variable_name in variable_names
        } for variable_type, variable_names in history_variables[backend].items()
    } for backend in ["be", "mlp"]
}

history_variable_trial_averages = {
    backend: {
        variable_type: {
            variable_name: np.mean( history_variables[backend][variable_type][variable_name], axis=0 )
                for variable_name in variable_names
        } for variable_type, variable_names in history_variables[backend].items()
    } for backend in ["be", "mlp"]
}

In [ ]:
# Compute variance (TODO)
history_variable_time_variances = {
    backend: {
        variable_type: {
            variable_name: np.mean( (history_variables[backend][variable_type][variable_name] - history_variable_time_averages[backend][variable_type][variable_name][:, np.newaxis])**2, axis=-1 )
                for variable_name in variable_names
        } for variable_type, variable_names in history_variables[backend].items()
    } for backend in ["be", "mlp"]
}

In [ ]:
# Compute RMSE for Trial vs. Trial and Averages
history_variable_rmse = {
    variable_type: {
        variable_name: np.sqrt(
            np.mean( (history_variables["be"][variable_type][variable_name] - history_variables["mlp"][variable_type][variable_name]) ** 2, axis=-1 ) )
            for variable_name in variable_names
    }
}

history_variable_trial_averages_rmse = {
    variable_type: {
        variable_name: np.sqrt(
            np.mean( (history_variable_trial_averages["be"][variable_type][variable_name] - history_variable_trial_averages["mlp"][variable_type][variable_name]) ** 2, axis=-1 ) )
            for variable_name in variable_names
    }
}

In [ ]:
from statsmodels.stats.weightstats import ttost_ind

import os
import sys
from contextlib import contextmanager


@contextmanager
def silence_output():
    # Route stdout and stderr to the system's null device
    new_target = open(os.devnull, "w")
    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = new_target, new_target
    try:
        yield
    finally:
        # Restore normal output handling
        sys.stdout, sys.stderr = old_stdout, old_stderr
        new_target.close()

def approximate_equivalence_p_test( be_data, mlp_data, label=None, units=None, tau=None, margin=None, relative_margin=0.01, relative_std=False, quiet=False ):
    if tau is None:
        tau = (
            int( emcee.autocorr.integrated_time(
                np.hstack( be_data.T,  ),
                           quiet=quiet
            )[0]),
            int( emcee.autocorr.integrated_time(
                np.hstack( mlp_data.T,  ),
                           quiet=quiet
            )[0]))

    # Here we are rejecting the null hypothesis of NON-equivalence

    # Run TOST
    # COME BACK HERE
    #if type( tau ) is tuple:
    #    strided_be_data  = np.concat( be_data[:5, ::tau[0]], axis=0 )
    #    strided_mlp_data = np.concat( be_data[5:, ::tau[1]], axis=0 )
    #else:
    #    strided_be_data  = np.concat( be_data[:5, ::tau[0]], axis=0 )
    #    strided_mlp_data = np.concat( be_data[5:, ::tau[1]], axis=0 )

    if type( tau ) is tuple:
        strided_be_data  = np.concat( be_data[:, ::tau[0]], axis=0 )
        strided_mlp_data = np.concat( be_data[:, ::tau[1]], axis=0 )
    else:
        strided_be_data  = np.concat( be_data[:, ::tau[0]], axis=0 )
        strided_mlp_data = np.concat( be_data[:, ::tau[1]], axis=0 )


    if margin is None:
        if not relative_std:
            mean   = np.mean( be_data )
            margin = mean*relative_margin
        else:
            variance = np.std( be_data )
            margin = variance*relative_margin

        margin_label = f"{100*relative_margin:.3f}%"
    else: 
        margin_label = f"{margin:.3f}{units}"

    p_value, t1, t2 = ttost_ind(
        x1=strided_be_data, 
        x2=strided_mlp_data, 
        low=-margin, 
        upp=margin, 
        usevar='unequal' # This makes it Welch's TOST i.e. we do not assume the same variance on each sample
    )

    if not quiet:
        if label is not None:
            label += " "
        print(f"{label}TOST p-value".ljust( 50 ) + f": p={p_value:.5f} on margin {margin_label} with tau={tau}")
    
    return p_value

def history_approximate_equivalence_p_test( history_data, variable_type, variable_name, **kwargs ):
    if "tau" not in kwargs.keys():
        kwargs["tau"] = (history_variable_taus[variable_type][variable_name], mlp_history_variable_taus[variable_type][variable_name])
    return approximate_equivalence_p_test(
        history_variables["be"][variable_type][variable_name],
        history_variables["mlp"][variable_type][variable_name],
        history_variable_labels[variable_name],
        history_variable_units[variable_name],
        **kwargs
    )
def history_per_layer_approximate_equivalence_p_test( history_data, variable_name, layer_window=[0, 128], margin=None, relative_margin=0.01, **kwargs ):
    if "tau" not in kwargs.keys():
        dynamic_tau = True

    p_values = []
    for i in range( *layer_window ):
        if dynamic_tau:
            kwargs["tau"] = (history_variable_taus["per_layer"][variable_name][i], mlp_history_variable_taus["per_layer"][variable_name][i])
        p_values.append( approximate_equivalence_p_test(
            history_variables["be"]["per_layer"][variable_name][:,..., i],
            history_variables["mlp"]["per_layer"][variable_name][:,..., i],
            history_variable_labels[variable_name],
            history_variable_units[variable_name],
            margin=margin,
            relative_margin=relative_margin,
            quiet=True,
            **kwargs
        ) )
    
    p_values = np.array( p_values )

    if margin is None:
        mean   = np.mean( history_variables["be"]["per_layer"][variable_name][..., i] )
        margin = mean*relative_margin

        margin_label = f"{100*relative_margin:.3f}%"
    else: 
        if margin > 1.0e-3:
            margin_label = f"{margin:.3f}{history_variable_units[variable_name]}"
        else:
            margin_label = f"{margin:.3e}{history_variable_units[variable_name]}"
    
    print(f"Worst {history_variable_labels[variable_name]} TOST p-value".ljust( 65 ) + f": p={max(p_values):.5f} on margin {margin_label}")

    failures = p_values > 0.05
    if np.any( failures ):
        print(f"Failures at:")
        print(np.where( failures ) )
    
    return p_values

In [ ]:
def to_sci_notation(value, precision=3):
    if not type( value )== str:
        return value
    if not "e+" in value and not "e-" in value:
        return value

    # value = value.replace( " ", "" )
    # Split the base from the exponent
    mantissa, exponent = value.split("e")


    switch = re.search( r'(?<=\d)[a-zA-Z() ]', exponent )

    if switch:
        switch_index = switch.start()
        units        = exponent[switch_index:]
        exponent     = exponent[:switch_index]
    else:
        units = ""
    
    
    # Casting to int automatically removes leading '+' and zeros (like '+08' to '8')
    exponent = exponent.lstrip( "0+" )

    # Exit if no exponent is left
    if len( exponent ) == 0:
        return f"${mantissa}{units}$"

    if exponent[0] == "-":
        exponent = "-" + exponent[1:].lstrip( "0+" )
    
    return f"${mantissa}\\times 10^{{{exponent}}} {units}$"
 



In [ ]:
import pandas as pd

per_layer_p_values      = {}
per_layer_margin_labels = {}


for variable_name in history_variable_names["per_layer"]:
    margin_type  = "absolute"
    margin_value = per_layer_dynamic_range_tost_tolerance * history_variable_ranges["per_layer"][variable_name]

    if margin_type == "absolute":
        p_values                     = history_per_layer_approximate_equivalence_p_test( history_variables, variable_name, margin=margin_value, layer_window=[0, 127] )
        per_layer_margin_labels[variable_name] = f"{margin_value:.2e} {history_variable_units[variable_name]}"
    elif margin_type == "relative":
        p_values                     = history_per_layer_approximate_equivalence_p_test( history_variables, variable_name, relative_margin=margin_value, layer_window=[0, 127] )
        per_layer_margin_labels[variable_name] = f"{100*margin_value:.3f}\\%"
    else:
        raise( ValueError( f"Incorrectly specified margin type for {variable_name}: {margin_type}" ) )

    per_layer_p_values[variable_name] = np.max( p_values )

#results = [
#    {
#        "Time Series"      : "$" + history_variable_labels[variable_name] + "$",
#        "Tolerance"        : per_layer_margin_labels[variable_name],
#        "Min $N_{eff}$"    : f"{np.min( 10*history_variables["be"]["per_layer"][variable_name].shape[1] // np.array( history_variable_taus["per_layer"][variable_name] ) ):.0f}",
#        "Max TOST p-value" : f"{np.max( per_layer_p_values[variable_name] ):.3e}",
#    } for variable_name in per_layer_margins.keys()
#]

results = [
    {
        "Variable"         : "$" + history_variable_labels[variable_name] + "$",
        "TOST Tolerance"   : per_layer_margin_labels[variable_name],
        "Min $N_{eff}$"        : f"{np.min( 10*history_variables["be"]["per_layer"][variable_name].shape[1] // np.array( history_variable_taus["per_layer"][variable_name] ) ):.0f}",
        "Max TOST p-value" : f"{np.max( per_layer_p_values[variable_name] ):.3e}"
    } for variable_name in history_variable_names["per_layer"]
]


df = pd.DataFrame( results )

df = df.map( to_sci_notation )

styled_table = (
    df.style
    .hide(axis="index")
)


display(styled_table)

with open("../paper_figures/time_series/per_layer_tost_table.tex", "w") as f:
    f.write(styled_table.to_latex(hrules=True))

In [ ]:
import pandas as pd
from IPython.display import display

p_test_table_keys = [
    ("scalar", "radavg"),
    ("averaged", "Tpmean"),
    ("scalar", "meanRH"),
    ("scalar", "tnumdrop")
]
scalar_margins = {
    "radavg"  : ("relative", 0.005 ),
    "Tpmean"  : ("absolute", 0.0075 ),
    "meanRH"  : ("absolute", 0.001 ),
    "tnumdrop": ("relative", 0.005 )
}
history_variable_formatters = {
    "radavg"   : ".2e",
    "Tpmean"   : ".3f",
    "meanRH"   : ".3f",
    "tnumdrop" : ".3e"
}

p_values = {}
margin_labels  = {}
for variable_type, variable_name in p_test_table_keys:
    #margin_type, margin_value = scalar_margins[variable_name]
    margin_type  = "absolute"
    margin_value = history_variable_ranges[variable_type][variable_name] * scalar_dynamic_range_tost_tolerance
    if margin_type == "absolute":
        #margin_labels[variable_name] = f"{margin_value:.2e} {history_variable_units[variable_name]}"
        margin_labels[variable_name] = f"{margin_value:.2e} {history_variable_units[variable_name]}"
        p_values[variable_name] = history_approximate_equivalence_p_test( history_variables, variable_type, variable_name, margin=margin_value )
    elif margin_type == "relative":
        margin_labels[variable_name] = f"{100*margin_value:.2f}\\%"
        p_values[variable_name] = history_approximate_equivalence_p_test( history_variables, variable_type, variable_name, relative_margin=margin_value )
    else:
        raise ValueError( "Unrecognized Margin Type for variable" + variable_name + ": " + margin_type )
    
#results = [
#    {
#        "Time Series"       : f"${history_variable_labels[variable_name]}$",
#        "$\mu_\\text{BE}$"  : f"{np.mean( history_variables['be'][variable_type][variable_name] ):{history_variable_formatters[variable_name]}} {history_variable_units[variable_name]}",
#        "$\mu_\\text{MLP}$" : f"{np.mean( history_variables['mlp'][variable_type][variable_name] ):{history_variable_formatters[variable_name]}} {history_variable_units[variable_name]}",
#        "Tolerance"         : margin_labels[variable_name],
#        "$N_{eff, \\text{BE}}$"    : 10*history_variables["be"][variable_type][variable_name].shape[1] // history_variable_taus[variable_type][variable_name],
#        "$N_{eff, \\text{MLP}}$"   : 10*history_variables["mlp"][variable_type][variable_name].shape[1] // mlp_history_variable_taus[variable_type][variable_name],
#        "TOST p-value"      : f"{p_values[variable_name]:.3e}",
#        "Result"            : "Pass" if p_values[variable_name] <= 0.05 else "Fail"
#    } for variable_type, variable_name in p_test_table_keys
#]

results = [
    {
        "Variable"               : f"${history_variable_labels[variable_name].replace( "}", "t}" )}$",
        "BE"                        : f"{np.mean( history_variables['be'][variable_type][variable_name] ):{history_variable_formatters[variable_name]}} {history_variable_units[variable_name]}",
        "MLP"                       : f"{np.mean( history_variables['mlp'][variable_type][variable_name] ):{history_variable_formatters[variable_name]}} {history_variable_units[variable_name]}",
        "TOST Tolerance"            : margin_labels[variable_name],
        "$N_{eff}$" : min( 10*history_variables["mlp"][variable_type][variable_name].shape[1] // mlp_history_variable_taus[variable_type][variable_name],10*history_variables["be"][variable_type][variable_name].shape[1] // history_variable_taus[variable_type][variable_name]),
        "TOST p-value"              : f"{p_values[variable_name]:.3e}"
        #"Result"            : "Pass" if p_values[variable_name] <= 0.05 else "Fail"
    } for variable_type, variable_name in p_test_table_keys
]


df = pd.DataFrame( results )
df = df.map( to_sci_notation )

styled_table = (
    df.style
    .format(formatter={
        "N_eff (Old)": "{:.0f}", 
        "N_eff (New)": "{:.0f}"
    })
    .hide(axis="index")
)

display(styled_table)

with open("../paper_figures/time_series/scalar_tost.tex", "w") as f:
    f.write(styled_table.to_latex(hrules=True))

In [ ]:
per_layer_to_plot = [
     "radmean", "Tpmean", "RHxym", "qstarm", "ql", "txym"
]


history_variable_labels["Tpmean"]  = "\\langle T_p\\rangle_{xyt}"

#for variable_name in history_variables["be"]["per_layer"].keys():
#    history_per_layer_ribbon_plot( history_variables, variable_name, [0, 127] )
#    plt.savefig( f"../paper_figures/per_layer/be_mlp_{variable_name}_per_layer.pdf",
#                 bbox_inches="tight",
#                 dpi=300 )
#
#    ax = history_per_layer_ribbon_plot( history_variables, variable_name, [0, 127] )
#    ax.get_legend().remove()


fig, ax_h = plt.subplots( nrows=3, ncols=2, figsize=(base_width*2,base_height*3) )

for index, variable_name in enumerate( per_layer_to_plot ):
     ax = ax_h.flat[index]
     history_per_layer_ribbon_plot( history_variables, variable_name, [0,127], ax=ax, rmse=True, rmse_relocate=True )
     plt.savefig( f"../paper_figures/per_layer/be_mlp_{variable_name}_per_layer.pdf",
                  bbox_inches="tight",
                  dpi=300 )
     if index != 1:
          ax.get_legend().remove()
     else:
          ax.get_legend().set_loc( "upper right" )

     ax.text(0.95, 0.10, f'({chr( ord( 'A' ) + index)})', transform=ax.transAxes, ha='right', va='center', weight='bold', fontsize=14)

plt.savefig( f"../paper_figures/per_layer/be_mlp_six_panel_per_layer.pdf",
             bbox_inches="tight",
             dpi=300 )


history_variable_labels["Tpmean"]  = "\\langle T_p\\rangle_{xyz}"

## Histogram Analysis

In [ ]:
HISTOGRAM_SAMPLE_COUNT             = 8
activated_particle_threshold_index = 206   # The index for the bin past which particles are considered activated
normalization_complete             = False # Whether they've already been scaled to PDFs

histogram_times = {
    "be": [file.variables["time"][:] for file in histogram_files["be"]],
    "mlp": [file.variables["time"][:] for file in histogram_files["mlp"]]
}

max_time = min( np.max( [timeline[-1] for timeline in histogram_times["be"]] ),
                np.max( [timeline[-1] for timeline in histogram_times["be"]] ) )

max_iteration = np.min( [np.searchsorted( timeline, max_time )
                        for timeline in [*histogram_times["be"],
                                         *histogram_times["mlp"]]] )

min_iteration = np.max( [np.searchsorted( timeline, steady_state_cutoff )
                         for timeline in [*histogram_times["be"],
                                          *histogram_times["mlp"]]] )

full_histogram_timeline    = histogram_times["be"][0]
trimmed_histogram_timeline = full_histogram_timeline[min_iteration:max_iteration]

histogram_indexes = np.linspace( 0, trimmed_histogram_timeline.shape[0] - 2, HISTOGRAM_SAMPLE_COUNT ).astype( "int" ) + min_iteration
histogram_times   = full_histogram_timeline[histogram_indexes]

histogram_variable_roots = [
    "res", "rad", "tp"
]
histogram_variable_names = [
    r + "hist" for r in histogram_variable_roots
]
histogram_bin_names = [
    r + "bins" for r in histogram_variable_roots
]
histogram_variable_labels    = {
    "radhist" : "\\langle r_p \\rangle_{xyzt}",
    "tphist"  : "\\langle T_p \\rangle_{xyzt}",
    "reshist" : "\\text{Residence Time}"
}
histogram_variable_units = {
    "radhist" : "\\mu m",
    "tphist"  : "K",
    "reshist" : "s"
}

In [ ]:
histogram_variables = {
    backend: {
        variable_name: np.array([
            file.variables[variable_name][min_iteration:max_iteration] for file in histogram_files[backend]
        ]) for variable_name in histogram_variable_names
    } for backend in ["be", "mlp"]
}
histogram_bins = {
    variable_name: histogram_files["be"][3].variables[variable_name]
    for variable_name in histogram_bin_names
}

In [ ]:
# Normalize Data into PDF is not already done...
if not normalization_complete:
    normalization_complete = True
    for backend in ["be", "mlp"]:
        for root in histogram_variable_roots:
            data = histogram_variables[backend][root + "hist"]
            bins = histogram_bins[root + "bins"]

            sum = np.sum( data, axis=-1 )[:, :, np.newaxis]  * np.abs( bins[1] - bins[0] )

            data /= sum

In [ ]:
histogram_variable_taus = {}
for backend in ["be", "mlp"]:
    histogram_variable_taus[backend] = {}
    for variable_name in histogram_variable_names:
        histogram_variable_taus[backend][variable_name] = np.nan_to_num( np.array([( np.ceil( compute_ensemble_integral_time_scale_fast(
                histogram_variables[backend][variable_name][..., i]
            ) ) )
            for i in range( histogram_variables[backend][variable_name].shape[2] )
        ]), nan=1.0 ).astype( "int" )

In [ ]:
histogram_variable_stds = {}

for backend in ["be", "mlp"]:
    histogram_variable_stds[backend] = {}
    for variable_name in histogram_variable_names:
        histogram_variable_stds[backend][variable_name] = np.array([
            np.std( histogram_variables[backend][variable_name][:, ::histogram_variable_taus[backend][variable_name][i], i], ddof=1 )
            for i in range( histogram_variables[backend][variable_name].shape[2] )
        ])

In [ ]:
histogram_variable_cis = {}
#histogram_variable_taus["be"]["radhist"][:] = 1

for variable_root in histogram_variable_roots:
    hist_name = variable_root + "hist"
    bin_name  = variable_root + "bins"

    histogram_variable_cis[hist_name] = []
    for bin_index in range( histogram_bins[bin_name].shape[0] ):
        be_data  = histogram_variables["be"][hist_name][:, ::histogram_variable_taus[backend][hist_name][bin_index], bin_index].flatten()
        mlp_data = histogram_variables["mlp"][hist_name][:, ::histogram_variable_taus[backend][hist_name][bin_index], bin_index].flatten()
        
        ci = stats.ttest_ind(mlp_data, be_data, equal_var=False).confidence_interval( confidence_level=0.95 )

        histogram_variable_cis[hist_name] += [[ci.low, ci.high]]

    histogram_variable_cis[hist_name] = np.array( histogram_variable_cis[hist_name] )

In [ ]:
def histogram_ribbon_plot( histogram_data, variable_root, bin_window=[0,10000], ax=None, log=False ):
    variable_name = variable_root + "hist"
    bins_name     = variable_root + "bins"

    be_data    = histogram_data["be"][variable_name][..., bin_window[0]:bin_window[1]].copy()
    mlp_data   = histogram_data["mlp"][variable_name][..., bin_window[0]:bin_window[1]].copy()
    data_label = "$" + histogram_variable_labels[variable_name] + "$"
    data_unit  = "$" + histogram_variable_units[variable_name] + "$"

    if "langle" in data_label:
        data_label = "$" + data_label[9:12] + "$"

    bins = histogram_bins[bins_name][bin_window[0]:bin_window[1]].copy()

    #be_data  /= np.sum( be_data, axis=-1 )[:, :, np.newaxis]  * np.abs( (bins[1] - bins[0]) )
    #mlp_data /= np.sum( mlp_data, axis=-1 )[:, :, np.newaxis] * np.abs( (bins[1] - bins[0]) )

    if log:
        bins = 10.0 ** bins

    ax = ribbon_plot( be_data, mlp_data, data_label, data_unit, ax=ax, indices=bins, std_a=histogram_variable_stds["be"][variable_name][..., bin_window[0]:bin_window[1]], std_b=histogram_variable_stds["mlp"][variable_name][..., bin_window[0]:bin_window[1]] )
    #ax.set_title( f"Time-Simulation Averaged Histogram Comparison for {data_label} in BE vs. MLP")
    ax.set_xlabel( f"{data_label} [{data_unit}]" )
    ax.set_ylabel( "Probability Density" )

    if log:
        ax.set_xscale( "log" )

    return ax

def histogram_ribbon_plot_error( histogram_data, variable_root, bin_window=[0,10000], ax=None, log=False ):
    variable_name = variable_root + "hist"
    bins_name     = variable_root + "bins"

    be_data    = histogram_data["be"][variable_name][..., bin_window[0]:bin_window[1]]
    mlp_data   = histogram_data["mlp"][variable_name][..., bin_window[0]:bin_window[1]]
    data_label = "$" + histogram_variable_labels[variable_name] + "$"
    data_unit  = "$" + histogram_variable_units[variable_name] + "$"

    if "langle" in data_label:
        data_label = "$" + data_label[9:12] + "$"

    bins = histogram_bins[bins_name][bin_window[0]:bin_window[1]] 

    #be_sum  = np.sum( be_data, axis=-1 )[:, :, np.newaxis]  * np.abs( bins[1] - bins[0] )
    #mlp_sum = np.sum( mlp_data, axis=-1 )[:, :, np.newaxis] * np.abs( bins[1] - bins[0] )

    #be_data  /= be_sum
    #mlp_data /= mlp_sum

    if log:
        bins = 10.0 ** bins

    ax = ribbon_plot_error( be_data, mlp_data, data_label, data_unit, ax=ax, indices=bins, ci=histogram_variable_cis[variable_name][bin_window[0]:bin_window[1]] )
    #ax.set_title( f"Time-Simulation Averaged Histogram Comparison for {data_label} in BE vs. MLP")
    ax.set_xlabel( f"{data_label} [{data_unit}]" )
    ax.set_ylabel( "Probability Density" )

    if log:
        ax.set_xscale( "log" )

    return ax

def histogram_std( histogram_data, variable_root, bin_window=[0, 10000] ):
    variable_name = variable_root + "hist"
    bins_name     = variable_root + "bins"

    be_data    = histogram_data["be"][variable_name][..., bin_window[0]:bin_window[1]]
    mlp_data   = histogram_data["mlp"][variable_name][..., bin_window[0]:bin_window[1]]

    be_data  = np.sum( be_data, axis=tuple( np.arange( be_data.ndim - 1 ) ) )
    mlp_data = np.sum( mlp_data, axis=tuple( np.arange( mlp_data.ndim - 1 ) ) )

    bins = histogram_bins[bins_name][bin_window[0]:bin_window[1]] 
    mids = (bins[1] - bins[0])/2 + bins

    be_mean = np.average( mids, weights=be_data )
    be_std  = np.sqrt( np.average( (mids - be_mean)**2, weights=be_data ) )

    mlp_mean = np.average( mids, weights=mlp_data )
    mlp_std  = np.sqrt( np.average( (mids - mlp_mean)**2, weights=mlp_data ) )

    return be_std, mlp_std

In [ ]:
initial_plots = [
    ("scalar", "radavg"),
    ("scalar", "meanRH"),
    ("averaged", "Tpmean"),
]

bin_windows = [
    [200, 1000],
    [100,375],
    [100,350]
]

fig, ax_h = plt.subplots( nrows=3, ncols=2, figsize=(base_width*2,base_height*3) )

r = 0

for row, root in enumerate( histogram_variable_roots ):
    if root == "rad":
        ax = histogram_ribbon_plot( histogram_variables, root, log=True, ax=ax_h[row][0], bin_window=bin_windows[row] )
    else:
        ax = histogram_ribbon_plot( histogram_variables, root, ax=ax_h[row][0], bin_window=bin_windows[row] )

    if row > 0:
        ax.get_legend().remove()
    else:
        ax.get_legend().set_loc( "upper left" )

    ax.text(0.12, 0.07, f'({chr( ord( 'A' ) + 2*row )})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)

    if root == "rad":
        ax = histogram_ribbon_plot_error( histogram_variables, root, ax=ax_h[row][1], log=True, bin_window=bin_windows[row] )
    else:
        ax = histogram_ribbon_plot_error( histogram_variables, root, ax=ax_h[row][1], bin_window=bin_windows[row] )

    ax.set_ylabel( f"Probability Density Error" )

    if row > 0:
            ax.get_legend().remove()
    else:
            ax.get_legend().set_loc( "upper left" )

    ax.text(0.12, 0.07, f'({chr( ord( 'A' ) + 2*row + 1 )})', transform=ax.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)


fig.align_ylabels()
plt.tight_layout()
plt.savefig( f"../paper_figures/histograms/rad_tmp_res_histogram.pdf",
            bbox_inches="tight",
             dpi=300 )


In [ ]:
be_timing_data  = np.fromfile( "paper_data/performance/be_timings.bin", dtype=np.float64 )
mlp_timing_data = np.fromfile( "paper_data/performance/mlp_timings.bin", dtype=np.float64 )

In [ ]:
mlp_timing_data.shape

In [ ]:
# There is exactly one iteration ~15x longer than the rest of the BE iterations which is stretching the x-axis.
# We can remove this without affecting the statistics

be_timing_data = be_timing_data[be_timing_data < 1.0e-3]
be_timing_data.shape

In [ ]:

color_be  = '#3498db'
color_mlp = '#e67e22'

histogram_fig = plt.figure( figsize=(base_size[0]*1.25, base_size[1]*1.25) )
ax1 = histogram_fig.gca()

plt.minorticks_on()
plt.grid( color="k", alpha=0.1 )

plt.hist( np.log10( be_timing_data ), bins=100, label="BE", color=color_be )
plt.hist( np.log10( mlp_timing_data ), bins=200, label="MLP", color=color_mlp )

plt.axvline(
    x=np.log10( np.mean( be_timing_data ) ),
    label=f"BE offline avg.",
    color="blue",
    linestyle="--"
)
plt.axvline(
    x=np.log10( np.mean( mlp_timing_data ) * 7.02 ),
    label=f"BE NTLP avg.",
    color="purple",
    linestyle="--"
)

plt.ylabel( "Count (#)" )
plt.xlabel( "Log Execution Time" )

plt.legend()

ax2 = plt.twinx()
ax2.hist( np.log10( be_timing_data ), bins=1000, color=color_be, alpha=0.2, label="BE CDF", cumulative="True", density=True )

plt.ylabel( "Cumulative Probability" )

ax1.legend( loc="upper left")
ax2.legend( loc="upper right" )

print( np.mean( be_timing_data*(10**6)))
print( np.mean( mlp_timing_data*(10**6)))
print( np.mean( mlp_timing_data*(10**6)*7.02))
print( np.mean( be_timing_data ) / np.mean( mlp_timing_data) )


plt.savefig( f"../paper_figures/performance/benchmark.pdf",
            bbox_inches="tight",
             dpi=300 )

In [ ]:
wasserstein_windows = {
    "residence time"       : ("res", "Residence Time", " s", [0, 1000]),
    "deactivated radius"   : ("rad", "Deactivated $\\log_{10} r_p$", "", [0, activated_particle_threshold_index]),
    "activated radius"     : ("rad", "Activated $\\log_{10} r_p$", "", [activated_particle_threshold_index, 1000]),
    "temperature"          : ("tp",  "$T_p$", " K", [0, 1000]),
}

wasserstein_distances = {
    key: wasserstein_distance( histogram_bins[root + "bins"][window[0]:window[1]], 
                                   histogram_bins[root + "bins"][window[0]:window[1]], 
                                   np.mean( histogram_variables["be"][root + "hist"], axis=(0,1) )[window[0]:window[1]],
                                   np.mean( histogram_variables["mlp"][root + "hist"], axis=(0,1) )[window[0]:window[1]] )
    for key, (root, _, _, window) in wasserstein_windows.items() 
}

histogram_standard_deviations = {
    key: histogram_std( histogram_variables, root, window )
    for key, (root, _, _, window) in wasserstein_windows.items()
}

In [ ]:
results = [
    {
        "Histogram"                     : label,
        "Wasserstein Metric (W)"        : f"{wasserstein_distances[key]:.3e}{units}",
        "Standard Deviation ($\\sigma$)": f"{histogram_standard_deviations[key][0]:.3e}{units}",
        "$W/\\sigma$"                   : f"{100*wasserstein_distances[key]/histogram_standard_deviations[key][0]:.3f}\\%"
    } for key, (_, label, units, _) in wasserstein_windows.items() 
]

df = pd.DataFrame( results )

df = df.map( to_sci_notation )

styled_table = (
    df.style
    .hide(axis="index")
)


display(styled_table)

with open("../paper_figures/histograms/wasserstein_table.tex", "w") as f:
    f.write(styled_table.to_latex(hrules=True))

## Radius Increment Analysis

In [ ]:
# Generate data
model, parameter_ranges, _, _ = load_model_checkpoint( model_path )
model_name = model.name()
# This is to make it work with the current model. Remove if using a new model
set_parameter_ranges( parameter_ranges )

time_point_count           = 300
droplet_radius_range       = (-6.75, -4.5)
temperature_sampling_range = (-3.0, 3.0)

droplet_count      = 20
results            = np.empty( (droplet_count, time_point_count, 2 ) )
model_results      = np.empty( (droplet_count, time_point_count, 2 ) )
radii              = 10 ** np.linspace( droplet_radius_range[0], droplet_radius_range[1], time_point_count )
droplet_parameters = np.empty( (droplet_count, time_point_count, 7) )

for i in range(droplet_count):
    #background_parameters = np.array( [ 290.0, 10**-17.66, 290.0, 1.04, 1.00, 0.1 ] )
    background_parameters  = np.hstack( [ scale_droplet_parameters( np.random.uniform( -1, 1, 6) )[1:], [0.1] ] )
    # Sample temperature
    background_parameters[0] = background_parameters[2] + np.random.uniform( *temperature_sampling_range )

    droplet_parameters[i]  = np.hstack( [ radii.reshape((-1, 1)), np.tile( background_parameters, (time_point_count, 1) ) ] )

model_results = do_inference( droplet_parameters[:, :, :-1].reshape( -1, 6 ), droplet_parameters[:, :, -1].reshape( -1 ), model, "cpu" ).reshape( droplet_count, time_point_count, 2)

strict_tolerances = True

for i in range(droplet_count):
    print(f"On Particle {i}")
    for count in range( time_point_count ):
        with warnings.catch_warnings():
            # This will force an exception anytime
            # BDF does something weird
            warnings.simplefilter("error")
            try:
                if strict_tolerances:
                    results[i, count, :] = solve_ivp( dydt, [0, 0.1], droplet_parameters[i, count, :2], atol=(1e-10, 1e-4), rtol=1.0e-7, method="BDF", t_eval=[droplet_parameters[i, count, -1]], args=(droplet_parameters[i, count, 2:-1],) ).y.T[:, :]
                else:
                    results[i, count, :] = solve_ivp( dydt, [0, 0.1], droplet_parameters[i, count, :2], method="BDF", t_eval=[droplet_parameters[i, count, -1]], args=(droplet_parameters[i, count, 2:-1],) ).y.T[:, :]
            except Exception as e:
                print(e)
                # If BDF failes, default to no change
                results[i,count,:] = droplet_parameters[i, count, :2]
                print(results[i,count,:])
                continue

In [ ]:
linear_radius_residuals = results[..., 0] - droplet_parameters[..., 0]

log_radius_in        = np.log10( droplet_parameters[..., 0] )
log_radius_out       = np.log10( results[..., 0] )
log_radius_residuals = log_radius_out - log_radius_in

quadratic_radius_in        = droplet_parameters[..., 0] ** 2
quadratic_radius_out       = results[..., 0] ** 2
quadratic_radius_residuals = quadratic_radius_out - quadratic_radius_in

#######

model_linear_radius_residuals = model_results[..., 0] - droplet_parameters[..., 0]

model_log_radius_in        = np.log10( droplet_parameters[..., 0] )
model_log_radius_out       = np.log10( model_results[..., 0] )
model_log_radius_residuals = model_log_radius_out - log_radius_in

model_quadratic_radius_in        = droplet_parameters[..., 0] ** 2
model_quadratic_radius_out       = model_results[..., 0] ** 2
model_quadratic_radius_residuals = quadratic_radius_out - quadratic_radius_in

#######

linear_radius_residuals_error    = model_linear_radius_residuals - linear_radius_residuals
log_radius_residuals_error       = model_log_radius_residuals - log_radius_residuals
quadratic_radius_residuals_error = model_quadratic_radius_residuals - quadratic_radius_residuals


In [ ]:
fig, ax_h = plt.subplots(2, 3, figsize=(6*3, 4.5*2), sharex=True, sharey="col" )

residual_data = [
    linear_radius_residuals*1.0e6,
    log_radius_residuals,
    quadratic_radius_residuals*1.0e12,
    model_linear_radius_residuals*1.0e6,
    model_log_radius_residuals,
    model_quadratic_radius_residuals*1.0e12
]
residual_labels = [
    "BDF",
    "BDF",
    "BDF",
    "MLP",
    "MLP",
    "MLP"
]
scale_descriptors = [
    "$\\Delta r_p \\ [\\mu m]$",
    "$\\Delta \\log_{10} r_p  \\ [-]$",
    "$\\Delta r_p^2 \\ [\\mu m^2]$",
    "$\\Delta r_p \\ [\\mu m]$",
    "$\\Delta \\log_{10} r_p  \\ [-]$",
    "$\\Delta r_p^2 \\ [\\mu m^2]$"
]

#fig.suptitle( "BDF Residual for Various Scales for {:d} Random Background Conditions with Variable Radius".format( droplet_count ),
#              size=14 )
for plot_index, axis in enumerate( ax_h.flat ):
    axis.minorticks_on()
    axis.grid( color="k", alpha=0.1 )
    #axis.set_title("{:s} Residual".format( residual_labels[plot_index] ))
    axis.set_xlabel("$\\log_{10} r_p$ [-]")
    axis.set_ylabel( "{:s}: {:s}".format( residual_labels[plot_index], scale_descriptors[plot_index]))
    axis.text(0.12, 0.05, f'({chr( ord( 'A' ) + plot_index)})', transform=axis.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)

    for droplet_index in range(droplet_count):
        axis.plot(log_radius_in[droplet_index], residual_data[plot_index][droplet_index, :], label=str(droplet_parameters[droplet_index,0, 1]) + str(droplet_parameters[droplet_index,0, 3]))

plt.tight_layout()

plt.savefig("../paper_figures/error_analysis_figures/varied_radius_increment_fig.png",
             bbox_inches="tight",
             dpi=300 )
plt.show()

In [ ]:
fig, ax_h = plt.subplots( 1, 3, figsize=(6*3, 4.5), sharex=True, sharey="col" )

residual_data = [
    linear_radius_residuals*1.0e6,
    log_radius_residuals,
    quadratic_radius_residuals*1.0e12
]
residual_labels = [
    "",
    "",
    ""
]
scale_descriptors = [
    "$\\Delta r_p \\ [\\mu m]$",
    "$\\Delta \\log_{10} r_p  \\ [-]$",
    "$\\Delta r_p^2 \\ [\\mu m^2]$"
]

#fig.suptitle( "BDF Residual for Various Scales for {:d} Random Background Conditions with Variable Radius".format( droplet_count ),
#              size=14 )
for plot_index, axis in enumerate( ax_h.flat ):
    axis.minorticks_on()
    axis.grid( color="k", alpha=0.1 )
    #axis.set_title("{:s} Residual".format( residual_labels[plot_index] ))
    axis.set_xlabel("$\\log_{10} r_p$ [-]")
    axis.set_ylabel( "{:s}{:s}".format( residual_labels[plot_index], scale_descriptors[plot_index]))
    axis.text(0.12, 0.05, f'({chr( ord( 'A' ) + plot_index)})', transform=axis.transAxes, ha='right', va='bottom', weight='bold', fontsize=14)

    for droplet_index in range(droplet_count):
        axis.plot(log_radius_in[droplet_index], residual_data[plot_index][droplet_index, :], label=str(droplet_parameters[droplet_index,0, 1]) + str(droplet_parameters[droplet_index,0, 3]))

plt.tight_layout()

plt.savefig("../paper_figures/error_analysis_figures/varied_radius_increment_fig_short.png",
             bbox_inches="tight",
             dpi=300 )
plt.show()
